# Robust Local Evaluation Harness for Kaggriculture Agents

Welcome! This is not a strategy notebook. It is an **experimental harness for measuring whether strategy changes actually improve an agent**.

In simulation environments like Kaggriculture, a single episode can be a noisy measurement of performance. Running an agent once and seeing a higher score does not guarantee the strategy is better; repeated independent episodes help estimate average performance and observed score variance. 

This notebook provides a **robust, copy-pasteable local testing framework**. It helps you transition from **single-game testing** to **repeated experimental evaluation**. Every episode in the evaluation loop is a completely separate game, guaranteeing independent results.

### What this notebook does:
1. Sets up the official `kaggle_environments` API.
2. Runs multiple independent matches between agents.
3. Collects and summarizes win rates, mean scores, standard error, and decision times.
4. Visualizes the score distribution so you can properly evaluate variance.

Let's get started!

In [ ]:
import time
import copy
import numpy as np
import matplotlib.pyplot as plt
from kaggle_environments import make

## 1. Define the Agents
Below are two trivial agents. In a real workflow, you would replace these with your own agents (for example, comparing `my_agent_v1` against `my_agent_v2`).

* **Baseline Agent**: A completely passive agent that returns an empty action dictionary.
* **Challenger Agent**: A simple agent that checks if it has at least 10 money, and if so, buys exactly one WHEAT seed on every turn it can afford it.

In [ ]:
def baseline_agent(obs):
    # A completely passive agent
    return {"farmer": ["PASS"], "hands": [], "market": []}

def challenger_agent(obs):
    # A simple agent that buys one wheat seed every turn it can afford it
    p = obs["player"]
    farms = obs["farms"]
    me = farms[p]
    
    market_orders = []
    if me["money"] >= 10:
        market_orders.append(["BUY_SEED", "WHEAT", 1])
        
    return {"farmer": ["PASS"], "hands": [], "market": market_orders}

## 2. The Evaluation Harness

Here is the core testing function. It uses the official `env.run()` API to simulate the game. Each time `env.run()` is called, it correctly starts an independent episode from scratch.

**A Note on Observation Copying (`COPY_OBSERVATION`)**:
When testing locally in Python, `kaggle_environments` passes mutable objects (like the observation dictionary) by reference to your agent function. If your agent accidentally modifies this observation in-place (e.g., `obs["money"] -= 100` to track budgets), it creates a reference-mutation hazard that can silently corrupt the local engine state. 

* `COPY_OBSERVATION = False` (default): This is more representative of the agent function's own decision time, as it does not add any extra logic outside your agent.
* `COPY_OBSERVATION = True`: This adds defensive observation isolation by passing a deepcopy of the state to the agent, but it adds copying overhead to the measured time.

*Note: Neither mode perfectly represents the total Kaggle runtime environment or their specific internal timeout measurements.*

In [ ]:
# Configuration
NUM_EPISODES = 10
COPY_OBSERVATION = False

def evaluate_agents(agent1, agent2, num_episodes=10):
    env = make("kaggriculture")
    
    results = {
        "agent1_scores": [],
        "agent2_scores": [],
        "agent1_times": [],
        "agent2_times": [],
        "errors": []
    }
    
    print(f"Starting evaluation of {num_episodes} independent episodes...")
    
    for i in range(num_episodes):
        # We wrap the agents to measure decision time and optionally protect the engine state
        def wrapped_agent1(obs):
            safe_obs = copy.deepcopy(obs) if COPY_OBSERVATION else obs
            t0 = time.perf_counter()
            act = agent1(safe_obs)
            results["agent1_times"].append(time.perf_counter() - t0)
            return act
            
        def wrapped_agent2(obs):
            safe_obs = copy.deepcopy(obs) if COPY_OBSERVATION else obs
            t0 = time.perf_counter()
            act = agent2(safe_obs)
            results["agent2_times"].append(time.perf_counter() - t0)
            return act

        # Run the episode (this independently evaluates a full game)
        steps = env.run([wrapped_agent1, wrapped_agent2])
        final_state = steps[-1]
        
        # Check for environment errors exposed in the final state
        if final_state[0].status == "ERROR" or final_state[1].status == "ERROR":
            results["errors"].append((i, final_state[0].status, final_state[1].status))
            
        # The score is stored in the reward field
        r1 = final_state[0].reward if final_state[0].reward is not None else 0
        r2 = final_state[1].reward if final_state[1].reward is not None else 0
        
        results["agent1_scores"].append(r1)
        results["agent2_scores"].append(r2)
        
        print(f"Episode {i+1}/{num_episodes} Complete | Agent 1: {r1:.1f} | Agent 2: {r2:.1f}")
        
    return results

## 3. Run the Benchmark
Let's run the evaluation. 
* **10 episodes** is an excellent baseline for a quick smoke test.
* **More episodes** provide more stable estimates. The appropriate sample size depends heavily on your agent's observed score variance and your available compute time. If two strategies result in very similar scores, you will need a larger sample size to confidently prove which is better.

In [ ]:
results = evaluate_agents(challenger_agent, baseline_agent, num_episodes=NUM_EPISODES)

## 4. Statistical Summary
Averages are helpful, but standard deviation and standard error tell you how consistent your agent actually is.

*Note on Agent Decision Time*: This measures the execution time of your Python function locally. It is clearly distinguished from deepcopy overhead (if disabled) and is not identical to the total Kaggle environment engine runtime. Kaggle Environments measures agent execution duration independently and applies its own specific timeout rules.

In [ ]:
def print_summary(results):
    a1_scores = np.array(results["agent1_scores"])
    a2_scores = np.array(results["agent2_scores"])
    
    wins = np.sum(a1_scores > a2_scores)
    losses = np.sum(a1_scores < a2_scores)
    ties = np.sum(a1_scores == a2_scores)
    
    n = len(a1_scores)
    a1_sem = np.std(a1_scores, ddof=1) / np.sqrt(n) if n > 1 else 0
    a2_sem = np.std(a2_scores, ddof=1) / np.sqrt(n) if n > 1 else 0
    
    print("=" * 40)
    print("📊 EVALUATION SUMMARY")
    print("=" * 40)
    print(f"Total Episodes: {n}")
    print(f"Win/Loss/Tie: {wins}W - {losses}L - {ties}T")
    print(f"Win Rate: {(wins/n)*100:.1f}%\n")
    
    print("Agent 1 (Challenger) Stats:")
    print(f"  Mean Score:   {np.mean(a1_scores):.2f} (± {a1_sem:.2f} SEM)")
    print(f"  Median Score: {np.median(a1_scores):.2f}")
    print(f"  Std Dev:      {np.std(a1_scores, ddof=1) if n > 1 else 0:.2f}")
    print(f"  Min/Max:      {np.min(a1_scores):.2f} / {np.max(a1_scores):.2f}")
    print(f"  Average Agent Decision Time: {np.mean(results['agent1_times'])*1000:.2f} ms\n")
    
    print("Agent 2 (Baseline) Stats:")
    print(f"  Mean Score:   {np.mean(a2_scores):.2f} (± {a2_sem:.2f} SEM)")
    print(f"  Average Agent Decision Time: {np.mean(results['agent2_times'])*1000:.2f} ms\n")
    
    if results["errors"]:
        print(f"⚠️ WARNING: {len(results['errors'])} episodes ended in an ERROR state.")
    else:
        print("✅ No errors detected during evaluation.")

print_summary(results)

## 5. Visualizations
Visualizing the data helps make variance immediately obvious.

In [ ]:
def plot_scores_over_time(results):
    episodes = range(1, len(results["agent1_scores"]) + 1)
    
    plt.figure()
    plt.plot(episodes, results["agent1_scores"], marker='o', label='Agent 1 (Challenger)')
    plt.plot(episodes, results["agent2_scores"], marker='o', label='Agent 2 (Baseline)')
    plt.title("Scores per Episode")
    plt.xlabel("Episode")
    plt.ylabel("Final Score")
    plt.legend()
    plt.show()

plot_scores_over_time(results)

In [ ]:
def plot_score_distribution(results):
    plt.figure()
    plt.hist(results["agent1_scores"], alpha=0.7, bins=10, label='Agent 1')
    plt.hist(results["agent2_scores"], alpha=0.7, bins=10, label='Agent 2')
    plt.title("Score Distribution")
    plt.xlabel("Score")
    plt.ylabel("Frequency")
    plt.legend()
    plt.show()

plot_score_distribution(results)

## 6. Pre-submission Checklist

Before uploading your final agent to the competition, run it through this harness and verify:

- [ ] **Agent runs locally:** The test completes without Python exceptions.
- [ ] **No runtime errors:** The evaluation summary reports no environment errors.
- [ ] **Performance is consistent:** You have inspected the variance and run enough episodes to confidently estimate performance.
- [ ] **Final File Check:** The agent tested here exactly matches the file you intend to submit.
- [ ] **Official Rules:** You have checked the official Kaggriculture competition page for any specific submission constraints.